<a href="https://colab.research.google.com/github/jamnalpriyanka21-collab/Careflow/blob/main/Copy_of_01_data_understanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:

!pip install -q pm4py

In [7]:
from google.colab import files
import pandas as pd
import pm4py

uploaded = files.upload()          # .xes.gz file select karo
path = list(uploaded.keys())[0]
log = pm4py.read_xes(path)
df = log if isinstance(log, pd.DataFrame) else pm4py.convert_to_dataframe(log)

print("Shape (rows, columns):", df.shape)



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




Saving Sepsis Cases - Event Log (2).xes.gz to Sepsis Cases - Event Log (2).xes (2).gz


/usr/local/lib/python3.13/dist-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/1050 [00:00<?, ?it/s]

Shape (rows, columns): (15214, 32)


In [8]:
CASE, ACT, TS = "case:concept:name", "concept:name", "time:timestamp"
raw = df.copy()   # raw ko kabhi modify nahi karenge

print("Rows:", len(raw), "| Cases:", raw[CASE].nunique(), "| Activities:", raw[ACT].nunique())

print("\n1) Missing in key columns:")
print(raw[[CASE, ACT, TS]].isna().sum())

print("\n2) Timestamp dtype:", raw[TS].dtype)
print("   Min:", raw[TS].min(), "| Max:", raw[TS].max())

if "lifecycle:transition" in raw.columns:
    print("\n3) Lifecycle values:")
    print(raw["lifecycle:transition"].value_counts(dropna=False))

in_order = raw.groupby(CASE)[TS].apply(lambda s: s.is_monotonic_increasing)
print("\n4) Cases NOT in time order in file:", (~in_order).sum(), "of", len(in_order))

print("5) Events sharing same case+timestamp:",
      raw.duplicated(subset=[CASE, TS], keep=False).sum())
print("6) Events with same case+activity+timestamp:",
      raw.duplicated(subset=[CASE, ACT, TS], keep=False).sum())
print("   Exact duplicate rows:", raw.duplicated().sum())

epc = raw.groupby(CASE).size()
print("\n7) Events per case:")
print(epc.describe())

srt = raw.sort_values([CASE, TS], kind="stable")
print("\n8) First activity per case:")
print(srt.groupby(CASE)[ACT].first().value_counts())
print("\n   Last activity per case:")
print(srt.groupby(CASE)[ACT].last().value_counts())

Rows: 15214 | Cases: 1050 | Activities: 16

1) Missing in key columns:
case:concept:name    0
concept:name         0
time:timestamp       0
dtype: int64

2) Timestamp dtype: datetime64[ns, UTC]
   Min: 2013-11-07 07:18:29+00:00 | Max: 2015-06-05 10:25:11+00:00

3) Lifecycle values:
lifecycle:transition
complete    15214
Name: count, dtype: int64

4) Cases NOT in time order in file: 0 of 1050
5) Events sharing same case+timestamp: 7526
6) Events with same case+activity+timestamp: 14
   Exact duplicate rows: 1

7) Events per case:
count    1050.000000
mean       14.489524
std        11.475941
min         3.000000
25%         9.000000
50%        13.000000
75%        16.000000
max       185.000000
dtype: float64

8) First activity per case:
concept:name
ER Registration     995
Leucocytes           18
IV Liquid            14
CRP                  10
ER Sepsis Triage      7
ER Triage             6
Name: count, dtype: int64

   Last activity per case:
concept:name
Release A           393
Retur